In [ ]:
# Import libraries
import os
import requests
import numpy as np
import os
import matplotlib.pyplot as plt
import pandas as pd
import rasterio
from scipy.stats import spearmanr # Using for Spearman Rank Correlation
import xarray as xr
import glob
from rasterio.mask import mask
from shapely.geometry import Point
from pyproj import Transformer
from shapely.geometry import box

In [7]:
et_gw_merged_all_sites = "/capstone/aridgw/data//openet_merged_timeseries_1km/merged_openet_data_1km.csv"
et_gw_merged_all_sites = pd.read_csv(et_gw_merged_all_sites)
et_gw_merged_all_sites.head()

,year_value,site_id,depth_to_gw_ft,depth_to_gw_m,latitude,longitude,data_source,region,bbox_side,open_et_version,scaled_annual_et_avg,mean_et,mean_precip,et_precip_ratio
0,2000,KSGS.371852100505801,239.39,72.966072,37.31502,-100.8505,USGS,Southern Kansas,1.0,2.0,781.134,797.106238,551.8382,1.444456
1,2001,KSGS.371852100505801,241.96,73.749408,37.31502,-100.8505,USGS,Southern Kansas,1.0,2.0,859.635,797.106238,551.8382,1.444456
2,2002,KSGS.371852100505801,242.78,73.999344,37.31502,-100.8505,USGS,Southern Kansas,1.0,2.0,787.151,797.106238,551.8382,1.444456
3,2003,KSGS.371852100505801,246.71,75.197208,37.31502,-100.8505,USGS,Southern Kansas,1.0,2.0,836.748,797.106238,551.8382,1.444456
4,2004,KSGS.371852100505801,247.71,75.502008,37.31502,-100.8505,USGS,Southern Kansas,1.0,2.0,666.882,797.106238,551.8382,1.444456


In [9]:
import re

def get_year(f):
    return int(re.findall(r"\d{4}", f)[0])

In [10]:
# extract yearly time series per site
from collections import defaultdict

def extract_yearly_variation(lon, lat, files, buffer_km=0.5):

    yearly_values = defaultdict(list)

    for f in files:
        year = get_year(f)

        with rasterio.open(f) as src:
            transformer = Transformer.from_crs("EPSG:4326", src.crs, always_xy=True)
            x, y = transformer.transform(lon, lat)

            half_size = buffer_km / 111.32
            geom = box(x-half_size, y-half_size, x+half_size, y+half_size)

            out, _ = mask(src, [geom], crop=True, all_touched=True)
            arr = out[0]

            yearly_values[year].append(np.nanmean(arr))

    return yearly_values

In [11]:
# compute variation per year
def summarize_yearly_variation(yearly_values):

    out = []

    for year, vals in yearly_values.items():
        vals = np.array(vals)

        mean = np.nanmean(vals)
        sd = np.nanstd(vals)

        if mean == 0 or np.isnan(mean):
            var = np.nan
        else:
            var = sd / mean

        out.append({
            "year_value": year,
            "precip_mean": mean,
            "precip_sd": sd,
            "precip_variation": var
        })

    return pd.DataFrame(out)

In [13]:
import glob

data_dir = "/capstone/aridgw/data/chirps_daily_data/"

files = sorted(glob.glob(data_dir + "*.tif"))
files = [f for f in files if any(str(year) in f for year in range(2000, 2021))]

print(len(files), "files found")

7671 files found


In [ ]:
# run for all sites
all_results = []

for lon, lat, site in zip(
    et_gw_merged_all_sites["longitude"],
    et_gw_merged_all_sites["latitude"],
    et_gw_merged_all_sites["site_id"]
):

    yearly_vals = extract_yearly_variation(lon, lat, files)

    df_site = summarize_yearly_variation(yearly_vals)
    df_site["site_id"] = site

    all_results.append(df_site)

precip_variation_df = pd.concat(all_results, ignore_index=True)

In [ ]:
from collections import defaultdict

# store values: (site_id, year) → list of daily values
data = defaultdict(list)

for f in files:
    
    year = get_year(f)

    with rasterio.open(f) as src:
        transformer = Transformer.from_crs("EPSG:4326", src.crs, always_xy=True)

        for lon, lat, site in zip(
            et_gw_merged_all_sites["longitude"],
            et_gw_merged_all_sites["latitude"],
            et_gw_merged_all_sites["site_id"]
        ):
            x, y = transformer.transform(lon, lat)

            half_size = 0.5 / 111.32
            geom = box(x-half_size, y-half_size, x+half_size, y+half_size)

            out, _ = mask(src, [geom], crop=True, all_touched=True)
            arr = out[0]

            data[(site, year)].append(np.nanmean(arr))